In [ ]:
from neo4j import GraphDatabase
import pandas as pd
import numpy as np

## Neo4j Connection Configuration

In [ ]:
URI = "bolt://localhost:7687" 
AUTH = ("neo4j", "") # change this to your Neo4j password
DATABASE_NAME = "" # change this to your Neo4j database name, or set to None for default database

In [ ]:
driver = GraphDatabase.driver(URI, auth=AUTH)

## Data Retrieval Query

In [ ]:
query_fetch = """
MATCH (u:User)
WHERE u.degreeScore IS NOT NULL
RETURN toString(u.id) AS id,  
       u.degreeScore AS degree, 
       u.pagerankScore AS pagerank, 
       u.betweennessScore AS betweenness
"""

print("Retrieving data from Neo4j...")

## Rank Aggregation and Database Update

In [ ]:
with driver.session(database=DATABASE_NAME) as session:
    result = session.run(query_fetch)
    data = [r.data() for r in result]
    
    if data:
        df = pd.DataFrame(data)
        N = len(df) 
        print(f"✅ Data loaded. Total Actors (N): {N}")
        
        # MATRIX NORMALIZATION 
        cols = ['degree', 'pagerank', 'betweenness']
        for col in cols:
            max_val = df[col].max()
            if max_val == 0:
                df[f'norm_{col}'] = 0
            else:
                df[f'norm_{col}'] = df[col] / max_val

        # DETERMINATION OF THE WEIGHT (WASPAS)
        w = 0.33  

        # WASPAS CALCULATION
        df['Q1'] = (df['norm_degree'] * w) + (df['norm_pagerank'] * w) + (df['norm_betweenness'] * w)
        df['Q2'] = (df['norm_degree'] ** w) * (df['norm_pagerank'] ** w) * (df['norm_betweenness'] ** w)
        
        lam = 0.5
        df['waspasScore'] = (lam * df['Q1']) + ((1-lam) * df['Q2'])

        # BORDA COUNT CALCULATION
        df['rank_deg'] = df['degree'].rank(ascending=False)
        df['rank_pr'] = df['pagerank'].rank(ascending=False)
        df['rank_bet'] = df['betweenness'].rank(ascending=False)
        
        df['point_deg'] = N - df['rank_deg']
        df['point_pr'] = N - df['rank_pr']
        df['point_bet'] = N - df['rank_bet']
        df['bordaScore'] = df['point_deg'] + df['point_pr'] + df['point_bet']

        # UPDATE DATABASE
        print("\nUpdating the database...")
        update_query = """
        UNWIND $data as row
        MATCH (u:User {id: toString(row.id)})
        SET u.waspasScore = toFloat(row.waspasScore),
            u.bordaScore = toFloat(row.bordaScore)
        RETURN count(u) as total
        """
        
        df['id'] = df['id'].astype(str)
        data_to_upload = df[['id', 'waspasScore', 'bordaScore']].to_dict('records')
        
        with driver.session(database=DATABASE_NAME) as session:
            batch_size = 1000
            total = 0
            for i in range(0, len(data_to_upload), batch_size):
                batch = data_to_upload[i:i+batch_size]
                res = session.run(update_query, data=batch)
                total += res.single()['total']
                
        print(f"Successfully updated {total} users.")
    else:
        print("No data available.")
    
driver.close()